# CV 融合算子教程

## 概述

本课程将系统讲解 CV（Cube-Vector）融合算子的开发与优化方法。

我们以昇腾 Ascend 910B（dav-2201 架构）/ Ascend 950PR/DT (dav-3510 架构)为硬件平台，围绕算子 **MmadLeakyReLU**，按以下路线逐步深入：

1. 先完成 **纯 Cube 算子（Mmad）** 与 **纯 Vector 算子（LeakyReLU）** 的独立实现；
2. 通过**分步调用**两个算子串联得到 MmadLeakyReLU 的计算结果；
3. 然后将两个算子**初步融合**，体验 CV 融合以及 AIC 与 AIV 之间的交互机制；
4. 最后进行**分块优化**，感受切分策略如何提高CV之间的并行度。

课程整体结构如下：
- 算子接口分析
- 分步多算子调用 —— Mmad + LeakyReLU
- CV 融合算子 —— MmadLeakyReLU
- 切分策略提高融合算子并行度
- 总结


---

# 1. 算子接口分析

MmadLeakyReLU 的计算分为两步：
  ```
  1. C = A * B + Bias           // 矩阵乘（Cube 侧）
  2. D = C > 0 ? C : C * alpha  // LeakyReLU（Vector 侧，alpha = 0.001）
  ```

第一步在 Cube 计算单元上完成矩阵乘，结果 C 传至 Vector 计算单元；第二步在 Vector 单元上完成逐元素的 LeakyReLU，最终输出 D。
下面，我们从纯 Cube 算子 Mmad 和纯 Vector 算子 LeakyReLU 的独立实现开始。

为便于上手，我们使用较小的矩阵规模 `m = n = k = 128`，无需切分即可一次性完成计算（每个计算单元都能完整装入输入数据），且算子仅在单个 AiCore 上执行，方便前后对比。参数配置如下：

**Mmad（纯 Cube）**

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <tr><td rowspan="1" align="center">算子类型(OpType)</td><td colspan="4" align="center">Mmad(纯Cube)</td></tr>
  <tr><td rowspan="4" align="center">算子输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
  <tr><td align="center">矩阵A</td><td align="center">[m, k] = [128, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td align="center">矩阵B</td><td align="center">[k, n] = [128, 128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td align="center">矩阵Bias</td><td align="center">[n] = [128]</td><td align="center">half</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">算子输出</td><td align="center">矩阵C</td><td align="center">[m, n] = [128, 128]</td><td align="center">float</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">核函数名</td><td colspan="4" align="center">mmad_custom</td></tr>
</table>
<br>

<img src="./images/04_04_cv_fusion/1_cube_mmad.png" alt="turing_test"  width="700px" >


**LeakyReLU（纯 Vector）**

Mmad 的输出矩阵 C 即为 LeakyReLU 的输入，LeakyReLU 的输出 D 即为最终的 MmadLeakyReLU 计算结果。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd;">
  <tr><td rowspan="1" align="center">算子类型(OpType)</td><td colspan="4" align="center">LeakyReLU(纯Vector)</td></tr>
  <tr><td rowspan="2" align="center">算子输入</td><td align="center">name</td><td align="center">shape</td><td align="center">data type</td><td align="center">format</td></tr>
  <tr><td align="center">C</td><td align="center">[m, n] = [128, 128]</td><td align="center">float</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">算子输出</td><td align="center">D</td><td align="center">[m, n] = [128, 128]</td><td align="center">float</td><td align="center">ND</td></tr>
  <tr><td rowspan="1" align="center">核函数名</td><td colspan="4" align="center">leakyrelu_custom</td></tr>
</table>

---

# 2. 分步多算子调用 —— Mmad + LeakyReLU

### 目录结构介绍

本课程代码按照如下目录结构存放：

```
├── mmad_leakyrelu
│   ├── scripts
│   │   ├── gen_data.py                // 输入数据和真值数据生成脚本
│   │   └── verify_result.py           // 验证输出数据和真值数据是否一致的验证脚本
│   ├── CMakeLists.txt                 // 编译工程文件
│   ├── data_utils.h                   // 数据读入写出函数
│   ├── mmad.h                         // mmad算子Kernel侧实现
│   ├── leakyrelu.h                    // leakyrelu算子Kernel侧实现
│   ├── mmad_leakyrelu.asc             // host侧调用
│   └── run.sh                         // 运行脚本
```
注意：为了代码组织清晰，Kernel 实现独立放在 mmad.h 和 leakyrelu.h 中。


In [ ]:
!mkdir src/mmad_leakyrelu

## 2.1 纯 Cube 算子 —— Mmad（矩阵乘法）

### Kernel 入口

```cpp
__global__ __cube__ void mmad_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c)
```
使用 `__cube__` 声明，表示这是一个纯 Cube 算子，仅在 AIC 上运行。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr><td rowspan="1" align="center">核函数入口参数</td><td colspan="4" align="center">含义</td></tr>
  <tr><td rowspan="1" align="left">a</td><td colspan="4" align="left">输入矩阵A GM地址</td></tr>
  <tr><td rowspan="1" align="left">b</td><td colspan="4" align="left">输入矩阵B GM地址</td></tr>
  <tr><td rowspan="1" align="left">bias</td><td colspan="4" align="left">输入矩阵Bias GM地址</td></tr>
  <tr><td rowspan="1" align="left">c</td><td colspan="4" align="left">输出矩阵C GM地址</td></tr>
</table>
<br>

该算子主要包含以下流水线阶段：

<div style="text-align: left; float: left;">
<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">阶段</td>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">方法</td>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">功能</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">输入数据搬入</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>CopyIn()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将矩阵 A、矩阵 B、矩阵Bias从 GM 搬运到 L1，矩阵 A、矩阵 B 均进行 ND → Nz 格式转换，矩阵Bias保持 ND 格式</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">矩阵A格式转换</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>SplitA()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将矩阵 A 从 L1 搬运到 L0A，dav-2201进行 Nz → Zz 格式转换，dav-3510保持Nz格式</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">矩阵B格式转换</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>SplitB()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将矩阵 B 从 L1 搬运到 L0B，进行 Nz → Zn 格式转换（带转置）</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">矩阵Bias搬运</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>SplitBias()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将矩阵 Bias 从 L1 搬运到 BT，half → float 类型转换（通过 <code>DataCopy</code>）</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">矩阵计算</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>Compute()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">核心矩阵计算 <code>C = A × B + Bias</code>，结果 C 为 Nz 格式</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">计算结果搬出</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>CopyOut()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将矩阵 C 从 L0C 搬运到 GM，Nz 格式转回 ND 格式</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

<br>
<img src="./images/04_04_cv_fusion/2_1_cube_route.png" alt="turing_test"  width="700px" >

我们先完成mmad.h, 填充矩阵计算的实现:

In [ ]:
%%writefile src/mmad_leakyrelu/mmad.h
#ifndef MMAD_H
#define MMAD_H

#include "kernel_operator.h"

// half type, cube block: [16, 16]
constexpr uint32_t CUBE_BLOCK = 16;
constexpr uint32_t CUBE_BLOCK_SIZE = 16 * 16;

class KernelMmad {
public:
    __aicore__ inline KernelMmad() {}

    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c)
    {
        aGM.SetGlobalBuffer((__gm__ half *)a);
        bGM.SetGlobalBuffer((__gm__ half *)b);
        cGM.SetGlobalBuffer((__gm__ float *)c);
        biasGM.SetGlobalBuffer((__gm__ half *)bias);

        uint32_t aSize = m * k * sizeof(half);
        uint32_t bSize = k * n * sizeof(half);
        uint32_t cSize = m * n * sizeof(float);
        pipe.InitBuffer(inQueueA1, 1, aSize);               // GM -> L1  矩阵A      [m, k]
        pipe.InitBuffer(inQueueA2, 1, aSize);               // L1 -> L0A 矩阵A      [m, k]
        pipe.InitBuffer(inQueueB1, 1, bSize);               // GM -> L1  矩阵B      [k, n]
        pipe.InitBuffer(inQueueB2, 1, bSize);               // L1 -> L0B 矩阵B      [k, n]
        pipe.InitBuffer(outQueueCO1, 1, cSize);             // L0A + L0B + BT => LOC 矩阵计算结果[m, n]
        pipe.InitBuffer(inQueueC1, 1, n * sizeof(half));    // GM -> L1  矩阵Bias   [n]
        pipe.InitBuffer(outQueueC2, 1, n * sizeof(float));  // L1 -> BT  矩阵Bias   [n]
    }

    __aicore__ inline void Process()
    {
        CopyIn();    // GM -> L1               搬运矩阵A, 矩阵B, 矩阵Bias
        SplitA();    // L1 -> L0A              搬运矩阵A
        SplitB();    // L1 -> L0B              搬运矩阵B
        SplitBias(); // L1 -> BT               搬运矩阵Bias
        Compute();   // L0A + L0B + BT -> L0C  进行矩阵计算
        CopyOut();   // L0C -> GM              搬运矩阵计算结果
    }

private:
    __aicore__ inline uint32_t CeilCubeBlock(uint32_t len)
    {
        return (len + CUBE_BLOCK - 1) / CUBE_BLOCK;
    }

    __aicore__ inline void CopyIn()
    {
        AscendC::LocalTensor<half> a1Local = inQueueA1.AllocTensor<half>();
        AscendC::LocalTensor<half> b1Local = inQueueB1.AllocTensor<half>();
        AscendC::LocalTensor<half> bias1Local = inQueueC1.AllocTensor<half>();

        AscendC::Nd2NzParams nd2nzA1Params;
        nd2nzA1Params.ndNum = 1;
        nd2nzA1Params.nValue = m;
        nd2nzA1Params.dValue = k;
        nd2nzA1Params.srcNdMatrixStride = 0;
        nd2nzA1Params.srcDValue = k;
        nd2nzA1Params.dstNzC0Stride = CeilCubeBlock(m) * CUBE_BLOCK;
        nd2nzA1Params.dstNzNStride = 1;
        nd2nzA1Params.dstNzMatrixStride = 0;
        AscendC::DataCopy(a1Local, aGM, nd2nzA1Params);

        AscendC::Nd2NzParams nd2nzB1Params;
        nd2nzB1Params.ndNum = 1;
        nd2nzB1Params.nValue = k;
        nd2nzB1Params.dValue = n;
        nd2nzB1Params.srcNdMatrixStride = 0;
        nd2nzB1Params.srcDValue = n;
        nd2nzB1Params.dstNzC0Stride = CeilCubeBlock(k) * CUBE_BLOCK;
        nd2nzB1Params.dstNzNStride = 1;
        nd2nzB1Params.dstNzMatrixStride = 0;
        AscendC::DataCopy(b1Local, bGM, nd2nzB1Params);

        AscendC::DataCopy(bias1Local, biasGM, n);

        inQueueA1.EnQue(a1Local);
        inQueueB1.EnQue(b1Local);
        inQueueC1.EnQue(bias1Local);
    }

    __aicore__ inline void SplitA()
    {
        AscendC::LocalTensor<half> a1Local = inQueueA1.DeQue<half>();
        AscendC::LocalTensor<half> a2Local = inQueueA2.AllocTensor<half>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // Nz -> Nz, __NPU_ARCH__为3510时，要求L0A上数据排布为：Nz
        AscendC::LoadData2DParamsV2 loadDataParams;
        loadDataParams.mStartPosition = 0;
        loadDataParams.kStartPosition = 0;
        loadDataParams.mStep = CeilCubeBlock(m);
        loadDataParams.kStep = CeilCubeBlock(k);
        loadDataParams.srcStride = CeilCubeBlock(m);
        loadDataParams.dstStride = CeilCubeBlock(m);
        loadDataParams.ifTranspose = false;
        AscendC::LoadData(a2Local, a1Local, loadDataParams);
#else
        // Nz -> Zz, __NPU_ARCH__为2201时，要求L0A上数据排布为：Zz
        uint32_t dstOffset = CeilCubeBlock(k) * CUBE_BLOCK_SIZE;
        uint32_t srcOffset = CUBE_BLOCK_SIZE;

        AscendC::LoadData2DParams loadDataParams;
        loadDataParams.repeatTimes = CeilCubeBlock(k);
        loadDataParams.srcStride = CeilCubeBlock(m);
        loadDataParams.dstGap = 0;
        loadDataParams.ifTranspose = false;

        // 一行一行分形的遍历搬运
        for (int i = 0; i < CeilCubeBlock(m); ++i) {
            AscendC::LoadData(a2Local[i * dstOffset], a1Local[i * srcOffset], loadDataParams);
        }
#endif

        inQueueA2.EnQue<half>(a2Local);
        inQueueA1.FreeTensor(a1Local);
    }
    __aicore__ inline void SplitB()
    {
        AscendC::LocalTensor<half> b1Local = inQueueB1.DeQue<half>();
        AscendC::LocalTensor<half> b2Local = inQueueB2.AllocTensor<half>();

        uint32_t dstOffset = CUBE_BLOCK_SIZE;
        uint32_t srcOffset = CeilCubeBlock(k) * CUBE_BLOCK_SIZE;

        AscendC::LoadData2DParams loadDataParams;
        loadDataParams.repeatTimes = CeilCubeBlock(k);
        loadDataParams.srcStride = 1;
        loadDataParams.dstGap = CeilCubeBlock(n) - 1;
        loadDataParams.ifTranspose = true;

        // 一列一列分型的遍历搬运
        for (int i = 0; i < CeilCubeBlock(n); ++i) {
            AscendC::LoadData(b2Local[i * dstOffset], b1Local[i * srcOffset], loadDataParams);
        }

        inQueueB1.FreeTensor(b1Local);
        inQueueB2.EnQue<half>(b2Local);
    }
    __aicore__ inline void SplitBias()
    {
        AscendC::LocalTensor<half> bias1Local = inQueueC1.DeQue<half>();
        AscendC::LocalTensor<float> bias2Local = outQueueC2.AllocTensor<float>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // __NPU_ARCH__为3510时，blockLen单位为32B
        AscendC::DataCopy(bias2Local, bias1Local, { 1, (uint16_t)(n * sizeof(float) / 32), 0, 0 });
#else
        // __NPU_ARCH__为2201时，blockLen单位为64B
        AscendC::DataCopy(bias2Local, bias1Local, { 1, (uint16_t)(n * sizeof(float) / 64), 0, 0 });
#endif

        outQueueC2.EnQue<float>(bias2Local);
        inQueueC1.FreeTensor(bias1Local);
    }
    __aicore__ inline void Compute()
    {
        AscendC::LocalTensor<half> a2Local = inQueueA2.DeQue<half>();
        AscendC::LocalTensor<half> b2Local = inQueueB2.DeQue<half>();
        AscendC::LocalTensor<float> bias2Local = outQueueC2.DeQue<float>();
        AscendC::LocalTensor<float> c1Local = outQueueCO1.AllocTensor<float>();

        AscendC::MmadParams mmadParams;
        mmadParams.m = m;
        mmadParams.n = n;
        mmadParams.k = k;
        mmadParams.cmatrixInitVal = false;
        AscendC::Mmad(c1Local, a2Local, b2Local, bias2Local, mmadParams);

        outQueueCO1.EnQue<float>(c1Local);
        inQueueA2.FreeTensor(a2Local);
        inQueueB2.FreeTensor(b2Local);
        outQueueC2.FreeTensor(bias2Local);
    }
    __aicore__ inline void CopyOut()
    {
        AscendC::LocalTensor<float> c1Local = outQueueCO1.DeQue<float>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        AscendC::FixpipeParamsArch3510<AscendC::CO2Layout::ROW_MAJOR> fixpipeParams;
#else
        AscendC::FixpipeParamsV220 fixpipeParams;
        fixpipeParams.ndNum = 1;
        fixpipeParams.srcNdStride = 0;
        fixpipeParams.dstNdStride = 0;
#endif
        fixpipeParams.nSize = n;
        fixpipeParams.mSize = m;
        fixpipeParams.srcStride = CeilCubeBlock(m) * CUBE_BLOCK;
        fixpipeParams.dstStride = n;
        AscendC::Fixpipe(cGM, c1Local, fixpipeParams);

        outQueueCO1.FreeTensor(c1Local);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::QuePosition::A1, 1> inQueueA1;
    AscendC::TQue<AscendC::QuePosition::A2, 1> inQueueA2;
    AscendC::TQue<AscendC::QuePosition::B1, 1> inQueueB1;
    AscendC::TQue<AscendC::QuePosition::B2, 1> inQueueB2;
    AscendC::TQue<AscendC::QuePosition::CO1, 1> outQueueCO1;
    AscendC::TQue<AscendC::QuePosition::C1, 1> inQueueC1;
    AscendC::TQue<AscendC::QuePosition::C2, 1> outQueueC2;

    AscendC::GlobalTensor<half> aGM;
    AscendC::GlobalTensor<half> bGM;
    AscendC::GlobalTensor<float> cGM;
    AscendC::GlobalTensor<half> biasGM;

    uint16_t m = 128, k = 128, n = 128;
};

__global__ __cube__ void mmad_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c)
{
    KernelMmad op;
    op.Init(a, b, bias, c);
    op.Process();
}

#endif // MMAD_H

## 2.2 纯 Vector 算子 —— LeakyReLU（激活函数）

### Kernel 入口

```cpp
__global__ __vector__ void leakyrelu_custom(GM_ADDR mmadResGm, GM_ADDR leakyreluResGm, uint32_t numElem, float alpha)
```
使用 `__vector__` 声明，表示这是一个纯 Vector 算子，仅在 AIV 上运行。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr><td rowspan="1" align="center">核函数入口参数</td><td colspan="4" align="center">含义</td></tr>
  <tr><td rowspan="1" align="left">mmadResGm</td><td colspan="4" align="left">输入mmad计算结果 GM地址</td></tr>
  <tr><td rowspan="1" align="left">leakyreluResGm</td><td colspan="4" align="left">输出leakyrelu计算结果 GM地址</td></tr>
  <tr><td rowspan="1" align="left">numElem</td><td colspan="4" align="left">需要计算的元素个数 (= m * n = 128 * 128)</td></tr>
  <tr><td rowspan="1" align="left">alpha</td><td colspan="4" align="left">leakyrelu中的alpha参数</td></tr>
</table>
<br>

该算子主要包含以下流水线阶段：

<div style="text-align: left; float: left;">
<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">阶段</td>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">方法</td>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">功能</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">输入数据搬入</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>CopyIn()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将mmad计算结果从GM搬运到UB</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">矢量计算</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>Compute()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">进行Leakyrelu计算</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">计算结果搬出</td>
    <td style="padding: 8px; border: 1px solid #ddd;"><code>CopyOut()</code></td>
    <td style="padding: 8px; border: 1px solid #ddd;">将计算结果从UB搬运到GM</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

<br>

<img src="./images/04_04_cv_fusion/2_2_leakyrelu_route.png" alt="turing_test"  width="700px" >

下面填充 leakyrelu.h：

In [ ]:
%%writefile src/mmad_leakyrelu/leakyrelu.h
#ifndef LEAKYRELU_H
#define LEAKYRELU_H

#include "kernel_operator.h"

class KernelLeakyReLU {
public:
    __aicore__ inline KernelLeakyReLU() {}

    // numElem表示需要计算的元素个数
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR z, uint32_t numElem, float alpha)
    {
        this->numElem = numElem;
        this->leakyReluAlpha = alpha;
        mmadResGm.SetGlobalBuffer((__gm__ float *)x, this->numElem);
        leakyreluResGm.SetGlobalBuffer((__gm__ float *)z, this->numElem);
        pipe.InitBuffer(inQueueX, 1, this->numElem * sizeof(float));
        pipe.InitBuffer(outQueueZ, 1, this->numElem * sizeof(float));
    }
    __aicore__ inline void Process()
    {
        CopyIn();
        Compute();
        CopyOut();
    }

private:
    __aicore__ inline void CopyIn()
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        AscendC::DataCopy(xLocal, mmadResGm, this->numElem);
        inQueueX.EnQue(xLocal);
    }
    __aicore__ inline void Compute()
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
        AscendC::LeakyRelu(zLocal, xLocal, this->leakyReluAlpha, this->numElem);
        outQueueZ.EnQue<float>(zLocal);
        inQueueX.FreeTensor(xLocal);
    }
    __aicore__ inline void CopyOut()
    {
        AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
        AscendC::DataCopy(leakyreluResGm, zLocal, this->numElem);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueueZ;
    AscendC::GlobalTensor<float> mmadResGm;
    AscendC::GlobalTensor<float> leakyreluResGm;
    uint32_t numElem;
    float leakyReluAlpha;
};

__global__ __vector__ void leakyrelu_custom(GM_ADDR mmadResGm, GM_ADDR leakyreluResGm, uint32_t numElem, float alpha)
{
    KernelLeakyReLU op;
    op.Init(mmadResGm, leakyreluResGm, numElem, alpha);
    op.Process();
}

#endif // LEAKYRELU_H

## 2.3 Host侧代码 mmad_leakyrelu.asc + data_utils.h

2.1和2.2中，Cube侧和Vector侧Kernel主体逻辑已经完成。现在我们要从host侧依次发起对device侧kernel的调用：先执行 `mmad_custom` 完成矩阵乘，结果存入 `cDevice`；再将 `cDevice` 作为输入传递至 `leakyrelu_custom`，完成整个 MmadLeakyReLU 的端到端计算。

```cpp
// mmad_leakyrelu.asc
mmad_custom<<<numBlocks, nullptr, stream>>>(aDevice, bDevice, biasDevice, cDevice);            // 运行mmad计算，结果存储在cDevice中
CHECK_ACL(aclrtSynchronizeStream(stream));

leakyrelu_custom<<<numBlocks, nullptr, stream>>>(cDevice, leakyreluResDevice, M * N, alpha);   // 将cDevice作为输入，继续计算leakyrelu
CHECK_ACL(aclrtSynchronizeStream(stream));
```

In [ ]:
%%writefile src/mmad_leakyrelu/mmad_leakyrelu.asc
#include <iostream>

#include "acl/acl.h"
#include "mmad.h"
#include "leakyrelu.h"
#include "data_utils.h"

#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);

int32_t main(int32_t argc, char *argv[])
{
    uint32_t M = 128;
    uint32_t N = 128;
    uint32_t K = 128;
    size_t aFileSize = M * K * sizeof(half);
    size_t bFileSize = K * N * sizeof(half);
    size_t biasFileSize = N * sizeof(half);
    size_t cFileSize = M * N * sizeof(float);
    uint32_t numBlocks = 1;                      // 启动的Aicore核数
    float alpha = 0.001;

    CHECK_ACL(aclInit(nullptr));
    int32_t deviceId = 0;
    CHECK_ACL(aclrtSetDevice(deviceId));
    aclrtStream stream = nullptr;
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t *aHost;
    uint8_t *aDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&aHost), aFileSize));
    CHECK_ACL(aclrtMalloc((void **)&aDevice, aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/x1_gm.bin", aFileSize, aHost, aFileSize);
    CHECK_ACL(aclrtMemcpy(aDevice, aFileSize, aHost, aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *bHost;
    uint8_t *bDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&bHost), bFileSize));
    CHECK_ACL(aclrtMalloc((void **)&bDevice, bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/x2_gm.bin", bFileSize, bHost, bFileSize);
    CHECK_ACL(aclrtMemcpy(bDevice, bFileSize, bHost, bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *biasHost;
    uint8_t *biasDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&biasHost), biasFileSize));
    CHECK_ACL(aclrtMalloc((void **)&biasDevice, biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/bias_gm.bin", biasFileSize, biasHost, biasFileSize);
    CHECK_ACL(aclrtMemcpy(biasDevice, biasFileSize, biasHost, biasFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *cHost;
    uint8_t *cDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&cHost), cFileSize));
    CHECK_ACL(aclrtMalloc((void **)&cDevice, cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));

    uint8_t *leakyreluResHost;
    uint8_t *leakyreluResDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&leakyreluResHost), cFileSize));
    CHECK_ACL(aclrtMalloc((void **)&leakyreluResDevice, cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));

    mmad_custom<<<numBlocks, nullptr, stream>>>(aDevice, bDevice, biasDevice, cDevice);
    CHECK_ACL(aclrtSynchronizeStream(stream));

    leakyrelu_custom<<<numBlocks, nullptr, stream>>>(cDevice, leakyreluResDevice, M * N, alpha);
    CHECK_ACL(aclrtSynchronizeStream(stream));

    CHECK_ACL(aclrtMemcpy(leakyreluResHost, cFileSize, leakyreluResDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));
    WriteFile("./output/output.bin", leakyreluResHost, cFileSize);

    CHECK_ACL(aclrtFree(aDevice));
    CHECK_ACL(aclrtFreeHost(aHost));
    CHECK_ACL(aclrtFree(bDevice));
    CHECK_ACL(aclrtFreeHost(bHost));
    CHECK_ACL(aclrtFree(biasDevice));
    CHECK_ACL(aclrtFreeHost(biasHost));
    CHECK_ACL(aclrtFree(cDevice));
    CHECK_ACL(aclrtFreeHost(cHost));
    CHECK_ACL(aclrtFree(leakyreluResDevice));
    CHECK_ACL(aclrtFreeHost(leakyreluResHost));

    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}

data_utils.h用于从.bin文件中读取数据或生成.bin文件。

In [ ]:
%%writefile src/mmad_leakyrelu/data_utils.h
#ifndef DATA_UTILS_H
#define DATA_UTILS_H
#include <fcntl.h>
#include <sys/stat.h>
#include <unistd.h>
#include <fstream>

#define ERROR_LOG(fmt, args...) fprintf(stdout, "[ERROR]  " fmt "\n", ##args)

bool ReadFile(const std::string &filePath, size_t &fileSize, void *buffer, size_t bufferSize)
{
    struct stat sBuf;
    int fileStatus = stat(filePath.data(), &sBuf);
    if (fileStatus == -1) {
        ERROR_LOG("failed to get file");
        return false;
    }
    if (S_ISREG(sBuf.st_mode) == 0) {
        ERROR_LOG("%s is not a file, please enter a file", filePath.c_str());
        return false;
    }

    std::ifstream file;
    file.open(filePath, std::ios::binary);
    if (!file.is_open()) {
        ERROR_LOG("Open file failed. path = %s", filePath.c_str());
        return false;
    }

    std::filebuf *buf = file.rdbuf();
    size_t size = buf->pubseekoff(0, std::ios::end, std::ios::in);
    if (size == 0) {
        ERROR_LOG("file size is 0");
        file.close();
        return false;
    }
    if (size > bufferSize) {
        ERROR_LOG("file size is larger than buffer size");
        file.close();
        return false;
    }
    buf->pubseekpos(0, std::ios::in);
    buf->sgetn(static_cast<char *>(buffer), size);
    fileSize = size;
    file.close();
    return true;
}

/**
 * @brief Write data to file
 * @param [in] filePath: file path
 * @param [in] buffer: data to write to file
 * @param [in] size: size to write
 * @return write result
 */
bool WriteFile(const std::string &filePath, const void *buffer, size_t size)
{
    if (buffer == nullptr) {
        ERROR_LOG("Write file failed. buffer is nullptr");
        return false;
    }

    int fd = open(filePath.c_str(), O_RDWR | O_CREAT | O_TRUNC, S_IRUSR | S_IWRITE);
    if (fd < 0) {
        ERROR_LOG("Open file failed. path = %s", filePath.c_str());
        return false;
    }

    size_t writeSize = write(fd, buffer, size);
    (void)close(fd);
    if (writeSize != size) {
        ERROR_LOG("Write file Failed.");
        return false;
    }

    return true;
}
#endif // DATA_UTILS_H


## 2.4 python输入真值生成脚本和精度校验脚本

我们将python脚本统一放在scripts目录下。本样例的数据生成和结果校验脚本依赖 ml_dtypes，运行样例前需安装该依赖。

In [ ]:
!mkdir -p src/mmad_leakyrelu/scripts                                                # 生成scipts文件夹
!python3 -m pip install ml_dtypes -i https://pypi.tuna.tsinghua.edu.cn/simple       # 安装ml_dtypes

gen_data.py：用于生成输入矩阵A、矩阵B、矩阵Bias，预期输出golden的.bin文件。

In [ ]:
%%writefile src/mmad_leakyrelu/scripts/gen_data.py
import os
import numpy as np
import ml_dtypes

def leaky_relu(x, alpha=0.01):
    return np.where(x >= 0, x, alpha * x)

def gen_golden_data():
    m, n, k, is_bias = 128, 128, 128, True

    os.makedirs("input", exist_ok=True)
    os.makedirs("output", exist_ok=True)

    x1_gm = np.random.randint(-100, 100, [m, k]).astype(np.float16)
    x2_gm = np.random.randint(-100, 100, [k, n]).astype(np.float16)

    if is_bias:
        bias_gm = np.random.randint(-10, 10, [n]).reshape([n]).astype(np.float16)
        matmul_golden = np.matmul(x1_gm.astype(np.float32), x2_gm.astype(np.float32)).astype(np.float32) + bias_gm
        bias_gm.tofile("./input/bias_gm.bin")
    else:
        matmul_golden = np.matmul(x1_gm.astype(np.float32), x2_gm.astype(np.float32)).astype(np.float32)

    mmad_leakyrelu_golden = leaky_relu(matmul_golden, alpha=0.001)

    x1_gm.tofile("./input/x1_gm.bin")
    x2_gm.tofile("./input/x2_gm.bin")
    mmad_leakyrelu_golden.tofile("./output/golden.bin")

if __name__ == "__main__":
    gen_golden_data()

verify_result.py：用于将算子的计算结果和golden的.bin文件进行对比，校验精度是否符合预期。

In [ ]:
%%writefile src/mmad_leakyrelu/scripts/verify_result.py
import sys
import numpy as np
import ml_dtypes

RELATIVE_TOL = 1e-3
ABSOLUTE_TOL = 1e-3
ERROR_TOL = 1e-3


def verify_result(output, golden):
    output = np.fromfile(output, dtype=np.float16).reshape(-1)
    golden = np.fromfile(golden, dtype=np.float16).reshape(-1)

    # Get total number of elements compared
    total_elements = golden.size

    different_element_results = np.isclose(output.astype(np.float32),
                                           golden.astype(np.float32),
                                           rtol=RELATIVE_TOL,
                                           atol=ABSOLUTE_TOL,
                                           equal_nan=True)
    different_element_indexes = np.where(different_element_results == False)[0]

    # Get total number of errors
    error_count = different_element_indexes.size

    # Print total comparison count and error count
    print(f"Total elements compared: {total_elements}")
    print(f"Total error elements: {error_count}")

    for index in range(len(different_element_indexes)):
        real_index = different_element_indexes[index]
        golden_data = float(golden[real_index])
        output_data = float(output[real_index])
        print(
            "data index: %06d, expected: %-.9f, actual: %-.9f, rdiff: %-.6f" %
            (real_index, golden_data, output_data,
            abs(output_data - golden_data) / golden_data))
        if index == 100:
            break

    error_ratio = float(different_element_indexes.size) / golden.size
    print("error ratio: %.4f, tolerance: %.4f" % (error_ratio, ERROR_TOL))
    return error_ratio <= ERROR_TOL


if __name__ == '__main__':
    try:
        res = verify_result(sys.argv[1], sys.argv[2])
        if not res:
            raise ValueError("[ERROR] result error")
        else:
            print("test pass!")
    except Exception as e:
        print(e)
        sys.exit(1)

### 2.5 CMakeLists.txt 和run.sh
最后新增 CMakeLists.txt 编译出可执行文件。`CMAKE_ASC_ARCHITECTURES` 表示硬件平台（dav-2201 / dav-3510）。<br>

In [ ]:
%%writefile src/mmad_leakyrelu/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_RUN_MODE "npu" CACHE STRING "Run mode: npu, cpu, sim")
set(CMAKE_ASC_ARCHITECTURES "dav-2201" CACHE STRING "NPU architecture: dav-2201, dav-3510")

find_package(ASC REQUIRED)

project(kernel_samples LANGUAGES ASC CXX)

add_executable(mmad_leakyrelu
    mmad_leakyrelu.asc
)

target_compile_options(mmad_leakyrelu PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES}>
)

通过 `bash run.sh --npu-arch={硬件平台}` 封装端到端的编译 → 运行 → 精度校验流程：
```
bash run.sh --npu-arch=dav-2201
```

In [ ]:
%%writefile src/mmad_leakyrelu/run.sh
if [ $# -eq 0 ]; then
    echo "错误：请提供 --npu-arch=xxx 参数"
    echo "用法: bash run.sh --npu-arch=dav-2201 或 bash run.sh --npu-arch=dav-3510"
    exit 1
fi

# 解析 --npu-arch=xxx 参数, 提取架构名
for arg in "$@"; do
    case "$arg" in
        --npu-arch=*)
            npu_arch="${arg#*=}"
            ;;
    esac
done

case "$npu_arch" in
    dav-2201|dav-3510)
        ;;
    *)
        echo "错误：不支持的架构名 '$npu_arch'，仅支持 dav-2201 或 dav-3510"
        exit 1
        ;;
esac

rm -rf build
mkdir -p build && cd build;                                               # 创建并进入build目录
cmake -DCMAKE_ASC_ARCHITECTURES="$npu_arch" ..;make -j;                   # 编译工程
python3 ../scripts/gen_data.py                                            # 生成输入数据、标杆产物
./mmad_leakyrelu                                                     # 执行可执行产物
python3 ../scripts/verify_result.py output/output.bin output/golden.bin   # 验证输出结果是否正确，确认算法逻辑正确

此时已完成所有文件的修改，执行 run.sh 验证分步调用算子的计算结果是否正确。

In [ ]:
# --npu-arch可以配置为dav-2201或dav-3510
!cd src/mmad_leakyrelu && bash run.sh --npu-arch=dav-2201

---

# 3. 融合算子 —— Mmad + LeakyReLU 混合编程

前面的分步调用存在明显缺陷：Cube 和 Vector 算子是**完全串行**的，Cube 侧必须全部计算完成后 Vector 侧才能启动，无法发挥两个计算单元的并行能力。<br>
我们先实现一个基础的 CV 融合算子：将 Cube 侧和 Vector 侧的 Kernel 合并到一个 `__mix__` 入口中，借助 `CrossCoreSetFlag` / `CrossCoreWaitFlag` 实现核间同步。当前版本不切分，目的是先理解 CV 交互机制；下一节将在此基础上引入切分以提升并行度。

核心改动是将 `mmad.h` 中的核函数入口改造为 `mmad_leakyrelu_custom`，实现 AIC 与 AIV 之间的交互。

### Kernel 入口

```cpp
__global__ __mix__(1,1) void mmad_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR leakyreluRes,
    uint32_t numElem, float alpha)
```
使用 `__mix__(1,1)` 声明，表示这是一个 mix 算子，AIC 与 AIV 配比为 1:1。`numBlocks = 1`，因此最终启动 **1 个 AIC + 1 个 AIV**——与前面一致：1个AIC 计算 Mmad，1个AIV 计算 LeakyReLU。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr><td rowspan="1" align="center">核函数入口参数</td><td colspan="4" align="center">含义</td></tr>
  <tr><td rowspan="1" align="left">a</td><td colspan="4" align="left">输入矩阵A GM地址</td></tr>
  <tr><td rowspan="1" align="left">b</td><td colspan="4" align="left">输入矩阵B GM地址</td></tr>
  <tr><td rowspan="1" align="left">bias</td><td colspan="4" align="left">输入矩阵Bias GM地址</td></tr>
  <tr><td rowspan="1" align="left">c</td><td colspan="4" align="left">矩阵计算结果C GM地址，作为leakyrelu的输入</td></tr>
  <tr><td rowspan="1" align="left">leakyreluRes</td><td colspan="4" align="left">输出leakyrelu计算结果 GM地址</td></tr>
  <tr><td rowspan="1" align="left">numElem</td><td colspan="4" align="left">需要计算的元素个数 (= m * n = 128 * 128)</td></tr>
  <tr><td rowspan="1" align="left">alpha</td><td colspan="4" align="left">leakyrelu中的alpha参数</td></tr>
</table>
<br>

### ASCEND_IS_AIC 和 ASCEND_IS_AIV 的作用

当算子同时涉及 AIC 和 AIV 的不同处理逻辑时，使用 `ASCEND_IS_AIC` / `ASCEND_IS_AIV` 宏做代码隔离，判断当前运行环境。被 `ASCEND_IS_AIC` 包裹的逻辑仅在 AIC 上执行，反之亦然。由此可将 Cube 侧和 Vector 侧逻辑清晰分开：AIC 只负责 `KernelMmad`，AIV 只负责 `KernelLeakyReLU`。

```cpp
// mmad.h
__global__ __mix__(1,1) void mmad_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR leakyreluRes,
    uint32_t numElem, float alpha)
{
    if ASCEND_IS_AIC {                // 仅在AIC中执行
        KernelMmad op;
        op.Init(a, b, bias, c);
        op.Process();
    } else { // ASCEND_IS_AIV         // 仅在AIV中执行 
        KernelLeakyReLU op;
        op.Init(c, leakyreluRes, numElem, alpha);
        op.Process();
    }
}
```

### AIC 和 AIV 之间的同步机制

Cube 和 Vector 是独立的计算单元，必须通过同步机制保证计算正确性：Vector 的计算必须在 Cube 完成后方可开始。我们使用 `CrossCoreSetFlag` / `CrossCoreWaitFlag` 来保障时序。核间同步场景，CrossCoreSetFlag和CrossCoreWaitFlag接口协同工作，前者用于通知调度模块"本核的指定pipe流水任务已经完成“，后者用于阻塞后续指令下发，直到所有相关核均完成同步后方可解除阻塞。

```cpp
template <uint8_t modeId, pipe_t pipe>
__aicore__ inline void CrossCoreSetFlag(uint16_t flagId);

template <uint8_t modeId = 0, pipe_t pipe = PIPE_S>
__aicore__ inline void CrossCoreWaitFlag(uint16_t flagId);
```

第二节的矩阵计算提到，矩阵 C 的计算结果通过 **Fixpipe** 从 L0C 搬运到 GM（流水为PIPE_FIX）。因此正确的时序是：**PIPE_FIX 流水完成后**，AIC 通过 `CrossCoreSetFlag` 通知 AIV 可从 GM 取数据计算；AIV 通过 `CrossCoreWaitFlag` 等待该通知。

> **注意**：这里未做切分，整体仅一次 L0C → GM 搬运（`op.Process()` 的最后一个步骤），因此直接在 AIC 侧 `op.Process()` 之后调用一次 `CrossCoreSetFlag` 即可；AIV 则在最开始就用 `CrossCoreWaitFlag` 等待。涉及切分时会有多次搬运，写法需调整，可参见下一节内容。

```cpp
// mmad_leakyrelu.asc
__global__ __mix__(1,1) void mmad_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR leakyreluRes,
    uint32_t numElem, float alpha)
{
    if ASCEND_IS_AIC {
        KernelMmad op;
        op.Init(a, b, bias, c);
        op.Process();
        AscendC::CrossCoreSetFlag<2, PIPE_FIX>(SYNC_AIC_AIV_FLAG); // 当Fixpipe(PIPE_FIX)完成后，AIC通知AIV。
    } else { // ASCEND_IS_AIV
        AscendC::CrossCoreWaitFlag(SYNC_AIC_AIV_FLAG);             // AIV等待AIC通知。直到通知以后才开始后续流程。
        KernelLeakyReLU op;
        op.Init(c, leakyreluRes, numElem, alpha);
        op.Process();
    }
}
```

总共需要修改两个文件：<br>
mmad.h 中的核函数入口变更为mmad_leakyrelu_custom。

In [ ]:
%%writefile src/mmad_leakyrelu/mmad.h
#ifndef MMAD_H
#define MMAD_H

#include "kernel_operator.h"
#include "leakyrelu.h"

// half type, cube block: [16, 16]
constexpr uint32_t CUBE_BLOCK = 16;
constexpr uint32_t CUBE_BLOCK_SIZE = 16 * 16;
constexpr uint16_t SYNC_AIC_AIV_FLAG = 0;      // Cube通知Vector C侧xx任务已经完成

class KernelMmad {
public:
    __aicore__ inline KernelMmad() {}

    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c)
    {
        aGM.SetGlobalBuffer((__gm__ half *)a);
        bGM.SetGlobalBuffer((__gm__ half *)b);
        cGM.SetGlobalBuffer((__gm__ float *)c);
        biasGM.SetGlobalBuffer((__gm__ half *)bias);

        uint32_t aSize = m * k * sizeof(half);
        uint32_t bSize = k * n * sizeof(half);
        uint32_t cSize = m * n * sizeof(float);
        pipe.InitBuffer(inQueueA1, 1, aSize);               // GM -> L1  矩阵A      [m, k]
        pipe.InitBuffer(inQueueA2, 1, aSize);               // L1 -> L0A 矩阵A      [m, k]
        pipe.InitBuffer(inQueueB1, 1, bSize);               // GM -> L1  矩阵B      [k, n]
        pipe.InitBuffer(inQueueB2, 1, bSize);               // L1 -> L0B 矩阵B      [k, n]
        pipe.InitBuffer(outQueueCO1, 1, cSize);             // L0A + L0B + BT => LOC 矩阵计算结果[m, n]
        pipe.InitBuffer(inQueueC1, 1, n * sizeof(half));    // GM -> L1  矩阵Bias   [n]
        pipe.InitBuffer(outQueueC2, 1, n * sizeof(float));  // L1 -> BT  矩阵Bias   [n]
    }

    __aicore__ inline void Process()
    {
        CopyIn();    // GM -> L1               搬运矩阵A，矩阵B，矩阵Bias
        SplitA();    // L1 -> L0A              搬运矩阵A
        SplitB();    // L1 -> L0B              搬运矩阵B
        SplitBias(); // L1 -> BT               搬运矩阵Bias
        Compute();   // L0A + L0B + BT -> L0C  进行矩阵计算
        CopyOut();   // L0C -> GM              搬运矩阵计算结果
    }

private:
    __aicore__ inline uint32_t CeilCubeBlock(uint32_t len)
    {
        return (len + CUBE_BLOCK - 1) / CUBE_BLOCK;
    }

    __aicore__ inline void CopyIn()
    {
        AscendC::LocalTensor<half> a1Local = inQueueA1.AllocTensor<half>();
        AscendC::LocalTensor<half> b1Local = inQueueB1.AllocTensor<half>();
        AscendC::LocalTensor<half> bias1Local = inQueueC1.AllocTensor<half>();

        AscendC::Nd2NzParams nd2nzA1Params;
        nd2nzA1Params.ndNum = 1;
        nd2nzA1Params.nValue = m;
        nd2nzA1Params.dValue = k;
        nd2nzA1Params.srcNdMatrixStride = 0;
        nd2nzA1Params.srcDValue = k;
        nd2nzA1Params.dstNzC0Stride = CeilCubeBlock(m) * CUBE_BLOCK;
        nd2nzA1Params.dstNzNStride = 1;
        nd2nzA1Params.dstNzMatrixStride = 0;
        AscendC::DataCopy(a1Local, aGM, nd2nzA1Params);

        AscendC::Nd2NzParams nd2nzB1Params;
        nd2nzB1Params.ndNum = 1;
        nd2nzB1Params.nValue = k;
        nd2nzB1Params.dValue = n;
        nd2nzB1Params.srcNdMatrixStride = 0;
        nd2nzB1Params.srcDValue = n;
        nd2nzB1Params.dstNzC0Stride = CeilCubeBlock(k) * CUBE_BLOCK;
        nd2nzB1Params.dstNzNStride = 1;
        nd2nzB1Params.dstNzMatrixStride = 0;
        AscendC::DataCopy(b1Local, bGM, nd2nzB1Params);

        AscendC::DataCopy(bias1Local, biasGM, n);

        inQueueA1.EnQue(a1Local);
        inQueueB1.EnQue(b1Local);
        inQueueC1.EnQue(bias1Local);
    }

    __aicore__ inline void SplitA()
    {
        AscendC::LocalTensor<half> a1Local = inQueueA1.DeQue<half>();
        AscendC::LocalTensor<half> a2Local = inQueueA2.AllocTensor<half>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // Nz -> Nz, __NPU_ARCH__为3510时，要求L0A上数据排布为：Nz
        AscendC::LoadData2DParamsV2 loadDataParams;
        loadDataParams.mStartPosition = 0;
        loadDataParams.kStartPosition = 0;
        loadDataParams.mStep = CeilCubeBlock(m);
        loadDataParams.kStep = CeilCubeBlock(k);
        loadDataParams.srcStride = CeilCubeBlock(m);
        loadDataParams.dstStride = CeilCubeBlock(m);
        loadDataParams.ifTranspose = false;
        AscendC::LoadData(a2Local, a1Local, loadDataParams);
#else
        // Nz -> Zz, __NPU_ARCH__为2201时，要求L0A上数据排布为：Zz
        uint32_t dstOffset = CeilCubeBlock(k) * CUBE_BLOCK_SIZE;
        uint32_t srcOffset = CUBE_BLOCK_SIZE;

        AscendC::LoadData2DParams loadDataParams;
        loadDataParams.repeatTimes = CeilCubeBlock(k);
        loadDataParams.srcStride = CeilCubeBlock(m);
        loadDataParams.dstGap = 0;
        loadDataParams.ifTranspose = false;

        // 一行一行分形的遍历搬运
        for (int i = 0; i < CeilCubeBlock(m); ++i) {
            AscendC::LoadData(a2Local[i * dstOffset], a1Local[i * srcOffset], loadDataParams);
        }
#endif

        inQueueA2.EnQue<half>(a2Local);
        inQueueA1.FreeTensor(a1Local);
    }
    __aicore__ inline void SplitB()
    {
        AscendC::LocalTensor<half> b1Local = inQueueB1.DeQue<half>();
        AscendC::LocalTensor<half> b2Local = inQueueB2.AllocTensor<half>();

        uint32_t dstOffset = CUBE_BLOCK_SIZE;
        uint32_t srcOffset = CeilCubeBlock(k) * CUBE_BLOCK_SIZE;

        AscendC::LoadData2DParams loadDataParams;
        loadDataParams.repeatTimes = CeilCubeBlock(k);
        loadDataParams.srcStride = 1;
        loadDataParams.dstGap = CeilCubeBlock(n) - 1;
        loadDataParams.ifTranspose = true;

        // 一列一列分型的遍历搬运
        for (int i = 0; i < CeilCubeBlock(n); ++i) {
            AscendC::LoadData(b2Local[i * dstOffset], b1Local[i * srcOffset], loadDataParams);
        }

        inQueueB1.FreeTensor(b1Local);
        inQueueB2.EnQue<half>(b2Local);
    }
    __aicore__ inline void SplitBias()
    {
        AscendC::LocalTensor<half> bias1Local = inQueueC1.DeQue<half>();
        AscendC::LocalTensor<float> bias2Local = outQueueC2.AllocTensor<float>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // __NPU_ARCH__为3510时，blockLen单位为32B
        AscendC::DataCopy(bias2Local, bias1Local, { 1, (uint16_t)(n * sizeof(float) / 32), 0, 0 });
#else
        // __NPU_ARCH__为2201时，blockLen单位为64B
        AscendC::DataCopy(bias2Local, bias1Local, { 1, (uint16_t)(n * sizeof(float) / 64), 0, 0 });
#endif
        outQueueC2.EnQue<float>(bias2Local);
        inQueueC1.FreeTensor(bias1Local);
    }
    __aicore__ inline void Compute()
    {
        AscendC::LocalTensor<half> a2Local = inQueueA2.DeQue<half>();
        AscendC::LocalTensor<half> b2Local = inQueueB2.DeQue<half>();
        AscendC::LocalTensor<float> bias2Local = outQueueC2.DeQue<float>();
        AscendC::LocalTensor<float> c1Local = outQueueCO1.AllocTensor<float>();

        AscendC::MmadParams mmadParams;
        mmadParams.m = m;
        mmadParams.n = n;
        mmadParams.k = k;
        mmadParams.cmatrixInitVal = false;
        AscendC::Mmad(c1Local, a2Local, b2Local, bias2Local, mmadParams);

        outQueueCO1.EnQue<float>(c1Local);
        inQueueA2.FreeTensor(a2Local);
        inQueueB2.FreeTensor(b2Local);
        outQueueC2.FreeTensor(bias2Local);
    }
    __aicore__ inline void CopyOut()
    {
        AscendC::LocalTensor<float> c1Local = outQueueCO1.DeQue<float>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        AscendC::FixpipeParamsArch3510<AscendC::CO2Layout::ROW_MAJOR> fixpipeParams;
#else
        AscendC::FixpipeParamsV220 fixpipeParams;
        fixpipeParams.ndNum = 1;
        fixpipeParams.srcNdStride = 0;
        fixpipeParams.dstNdStride = 0;
#endif
        fixpipeParams.nSize = n;
        fixpipeParams.mSize = m;
        fixpipeParams.srcStride = CeilCubeBlock(m) * CUBE_BLOCK;
        fixpipeParams.dstStride = n;
        AscendC::Fixpipe(cGM, c1Local, fixpipeParams);

        outQueueCO1.FreeTensor(c1Local);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::QuePosition::A1, 1> inQueueA1;
    AscendC::TQue<AscendC::QuePosition::A2, 1> inQueueA2;
    AscendC::TQue<AscendC::QuePosition::B1, 1> inQueueB1;
    AscendC::TQue<AscendC::QuePosition::B2, 1> inQueueB2;
    AscendC::TQue<AscendC::QuePosition::CO1, 1> outQueueCO1;
    AscendC::TQue<AscendC::QuePosition::C1, 1> inQueueC1;
    AscendC::TQue<AscendC::QuePosition::C2, 1> outQueueC2;

    AscendC::GlobalTensor<half> aGM;
    AscendC::GlobalTensor<half> bGM;
    AscendC::GlobalTensor<float> cGM;
    AscendC::GlobalTensor<half> biasGM;

    uint16_t m = 128, k = 128, n = 128;
};

__global__ __mix__(1,1) void mmad_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR leakyreluRes,
    uint32_t numElem, float alpha)
{
    if ASCEND_IS_AIC {
        KernelMmad op;
        op.Init(a, b, bias, c);
        op.Process();
        AscendC::CrossCoreSetFlag<2, PIPE_FIX>(SYNC_AIC_AIV_FLAG);
    } else { // ASCEND_IS_AIV
        AscendC::CrossCoreWaitFlag(SYNC_AIC_AIV_FLAG);
        KernelLeakyReLU op;
        op.Init(c, leakyreluRes, numElem, alpha);
        op.Process();
    }
}

#endif // MMAD_H

 mmad_leakyrelu.asc中的host侧调用需要修改为新写的mmad_leakyrelu_custom。

In [ ]:
%%writefile src/mmad_leakyrelu/mmad_leakyrelu.asc
#include <iostream>

#include "acl/acl.h"
#include "mmad.h"
#include "data_utils.h"

#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);

int32_t main(int32_t argc, char *argv[])
{
    uint32_t M = 128;
    uint32_t N = 128;
    uint32_t K = 128;
    size_t aFileSize = M * K * sizeof(half);
    size_t bFileSize = K * N * sizeof(half);
    size_t biasFileSize = N * sizeof(half);
    size_t cFileSize = M * N * sizeof(float);
    uint32_t numBlocks = 1;                      // 启动的Aicore核数
    float alpha = 0.001;

    CHECK_ACL(aclInit(nullptr));
    int32_t deviceId = 0;
    CHECK_ACL(aclrtSetDevice(deviceId));
    aclrtStream stream = nullptr;
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t *aHost;
    uint8_t *aDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&aHost), aFileSize));
    CHECK_ACL(aclrtMalloc((void **)&aDevice, aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/x1_gm.bin", aFileSize, aHost, aFileSize);
    CHECK_ACL(aclrtMemcpy(aDevice, aFileSize, aHost, aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *bHost;
    uint8_t *bDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&bHost), bFileSize));
    CHECK_ACL(aclrtMalloc((void **)&bDevice, bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/x2_gm.bin", bFileSize, bHost, bFileSize);
    CHECK_ACL(aclrtMemcpy(bDevice, bFileSize, bHost, bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *biasHost;
    uint8_t *biasDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&biasHost), biasFileSize));
    CHECK_ACL(aclrtMalloc((void **)&biasDevice, biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/bias_gm.bin", biasFileSize, biasHost, biasFileSize);
    CHECK_ACL(aclrtMemcpy(biasDevice, biasFileSize, biasHost, biasFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *cHost;
    uint8_t *cDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&cHost), cFileSize));
    CHECK_ACL(aclrtMalloc((void **)&cDevice, cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));

    uint8_t *leakyreluResHost;
    uint8_t *leakyreluResDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&leakyreluResHost), cFileSize));
    CHECK_ACL(aclrtMalloc((void **)&leakyreluResDevice, cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));

    mmad_leakyrelu_custom<<<numBlocks, nullptr, stream>>>(aDevice, bDevice, biasDevice, cDevice, leakyreluResDevice,
        M * N, alpha);
    CHECK_ACL(aclrtSynchronizeStream(stream));

    CHECK_ACL(aclrtMemcpy(leakyreluResHost, cFileSize, leakyreluResDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));
    WriteFile("./output/output.bin", leakyreluResHost, cFileSize);

    CHECK_ACL(aclrtFree(aDevice));
    CHECK_ACL(aclrtFreeHost(aHost));
    CHECK_ACL(aclrtFree(bDevice));
    CHECK_ACL(aclrtFreeHost(bHost));
    CHECK_ACL(aclrtFree(biasDevice));
    CHECK_ACL(aclrtFreeHost(biasHost));
    CHECK_ACL(aclrtFree(cDevice));
    CHECK_ACL(aclrtFreeHost(cHost));
    CHECK_ACL(aclrtFree(leakyreluResDevice));
    CHECK_ACL(aclrtFreeHost(leakyreluResHost));

    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}

用户可执行如下脚本，验证 CV 融合后产物精度是否符合预期：

> **提示**：可尝试同时注释掉 `CrossCoreSetFlag` 和 `CrossCoreWaitFlag`，体验缺少同步保障时的计算效果。

In [ ]:
# --npu-arch可以配置为dav-2201或dav-3510
!cd src/mmad_leakyrelu && bash run.sh --npu-arch=dav-2201

---

# 4. 切分策略提高融合算子并行度

虽然我们现在完成了基础融合，但 Cube 和 Vector 之间基本是串行的——Cube 整块算完 Vector 才启动，无法利用两个计算单元的并行能力。本节通过**切块计算**提升并行度，并借助流水图直观感受并行执行的效果。

`mmad.h` 和 `leakyrelu.h` 均需大量修改以支持切块搬运与计算。

### 分块方案

将矩阵 A 和矩阵 B 沿 M、N 轴各对半切分（K 轴不切），共产生 **4 个子块**：

<br>
<img src="./images/04_04_cv_fusion/4_mmad_split.png" alt="turing_test"  width="700px" >
<br>
<br>

每个 `[64, 64]` 子块依次执行 Cube（Mmad）→ Vector（LeakyReLU），Cube 侧每算完一个小块就立即通知 Vector 侧开始处理，形成**子块粒度的流水并行**。

### Kernel 入口

```cpp
__global__ __mix__(1,1) void mmad_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR leakyreluRes, LeakyReLUCustomTilingData tiling)
```

之前 LeakyReLU 是连续搬运，只需一个总数据量入参。切分后变为非连续搬运，需要 `baseM`、`baseN` 等参数来计算 stride 跳跃量。为此引入结构体 `LeakyReLUCustomTilingData` 统一传递这些参数。

<br>
<img src="./images/04_04_cv_fusion/4_leakyrelu_datacopy.png" alt="turing_test"  width="700px" >
<br>
<br>

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr><td rowspan="1" align="center">核函数入口参数</td><td colspan="4" align="center">含义</td></tr>
  <tr><td rowspan="1" align="left">a</td><td colspan="4" align="left">输入矩阵A GM地址</td></tr>
  <tr><td rowspan="1" align="left">b</td><td colspan="4" align="left">输入矩阵B GM地址</td></tr>
  <tr><td rowspan="1" align="left">bias</td><td colspan="4" align="left">输入矩阵Bias GM地址</td></tr>
  <tr><td rowspan="1" align="left">c</td><td colspan="4" align="left">矩阵计算结果C GM地址，作为leakyrelu的输入</td></tr>
  <tr><td rowspan="1" align="left">leakyreluRes</td><td colspan="4" align="left">输出leakyrelu计算结果 GM地址</td></tr>
  <tr><td rowspan="1" align="left">tiling</td><td colspan="4" align="left">结构体存储 m、n、k、baseM、baseN、alpha 等参数</td></tr>
</table>
<br>


新的结构体LeakyReLUCustomTilingData如下所示：
```cpp
struct LeakyReLUCustomTilingData
{
    uint32_t m;
    uint32_t n;
    uint32_t k;
    uint32_t baseM;
    uint32_t baseN;
    float alpha;
};
```

### 核间同步变更

切分后共需执行 2×2 轮计算，每轮使用 `mProgress` 和 `nProgress` 定位 GM 上的数据块。每完成一次分块的 `CopyOut`，立即调用 `CrossCoreSetFlag` 通知 Vector 侧并发处理，最大化并行度。

**Cube 侧（mmad.h）**

```cpp
// mmad.h
__aicore__ inline void Process()
{
    for (int mProgress = 0; mProgress < (m / baseM); mProgress++) {
        for (int nProgress = 0; nProgress < (n / baseN); nProgress++) {
            CopyIn(mProgress, nProgress);    // GM -> L1               搬运矩阵A，矩阵B, 矩阵Bias
            SplitA();                        // L1 -> L0A              搬运矩阵A
            SplitB();                        // L1 -> L0B              搬运矩阵B
            SplitBias();                     // L1 -> BT               搬运矩阵Bias
            Compute();                       // L0A + L0B + BT -> L0C  进行矩阵计算
            CopyOut(mProgress, nProgress);   // L0C -> GM              搬运矩阵计算结果

            AscendC::CrossCoreSetFlag<2, PIPE_FIX>(SYNC_AIC_AIV_FLAG);   // 计算完一次分块，通知Vector
        }
    }
}
```

**Vector 侧（leakyrelu.h）**

Vector 侧同样需执行 2×2 = 4 轮。每轮开始时通过 `CrossCoreWaitFlag` 等待 Cube 侧的通知，然后根据当前子块索引计算 GM 上的起始地址，配置对应的搬运 stride 参数。

```cpp
// leakyrelu.h
__aicore__ inline void Process()
{
    uint32_t loopNum = (this->m / this->baseM) * (this->n / this->baseN);
    for (int32_t i = 0; i < loopNum; i++) {
        AscendC::CrossCoreWaitFlag(SYNC_FLAG);
        // 一行的矩阵子块中，属于第几块（列方向）
        uint32_t cubePerRow = n / baseN;
        uint32_t rowIdx = i % cubePerRow;
        uint32_t rowStartIndex = baseN * rowIdx;
        // 多行的矩阵子块中，属于第几行（行方向）
        uint32_t wholeRowidx = i / cubePerRow;
        uint32_t gmIndex = wholeRowidx * baseM * n + rowStartIndex;
        CopyIn(i, gmIndex);
        Compute(i);
        CopyOut(i, gmIndex);
    }
}
```

总共需要修改三个个文件：<br>
· mmad.h 中的算子逻辑要为切分做适配<br>
· leakyrelu.h 中的算子逻辑要为切分做适配<br>
· mmad_leakyrelu.asc中的host侧需要稍作修改，因为需要传给核函数LeakyReLUCustomTilingData

In [ ]:
%%writefile src/mmad_leakyrelu/mmad.h
#ifndef MMAD_H
#define MMAD_H

#include "kernel_operator.h"
#include "leakyrelu.h"

// half type, cube block: [16, 16]
constexpr uint32_t CUBE_BLOCK = 16;
constexpr uint32_t CUBE_BLOCK_SIZE = 16 * 16;
constexpr uint16_t SYNC_AIC_AIV_FLAG = 0;      // Cube通知Vector C侧xx任务已经完成

class KernelMmad {
public:
    __aicore__ inline KernelMmad() {}

    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c)
    {
        // m=128, n=128, k=128. 因此原始 矩阵A [128, 128], 矩阵B [128， 128]
        // 现在将矩阵A和矩阵B对半进行切分(k轴不切)，即baseM = 64, baseN = 64, 即分成小块矩阵A[64, 128]，小块矩阵B[128, 64]
        // 因此总共要进行4次小块矩阵A和小块矩阵B的矩阵乘计算
        aGM.SetGlobalBuffer((__gm__ half *)a);
        bGM.SetGlobalBuffer((__gm__ half *)b);
        cGM.SetGlobalBuffer((__gm__ float *)c);
        biasGM.SetGlobalBuffer((__gm__ half *)bias);

        uint32_t baseASize = baseM * k * sizeof(half);
        uint32_t baseBSize = k * baseN * sizeof(half);
        uint32_t baseCSize = baseM * baseN * sizeof(float);
        pipe.InitBuffer(inQueueA1, 1, baseASize);      // GM -> L1  矩阵A      [baseM, k]
        pipe.InitBuffer(inQueueA2, 1, baseASize);      // L1 -> L0A 矩阵A      [baseM, k]
        pipe.InitBuffer(inQueueB1, 1, baseBSize);      // GM -> L1  矩阵B      [k, baseN]
        pipe.InitBuffer(inQueueB2, 1, baseBSize);      // L1 -> L0B 矩阵B      [k, baseN]
        pipe.InitBuffer(outQueueCO1, 1, baseCSize);    // L0A + L0B + BT => LOC 矩阵计算结果[baseM, baseN]
        pipe.InitBuffer(inQueueC1, 1, baseN);          // GM -> L1  矩阵Bias   [baseN]
        pipe.InitBuffer(outQueueC2, 1, baseN);         // L1 -> BT  矩阵Bias   [baseN]
    }

    __aicore__ inline void Process()
    {
        for (int mProgress = 0; mProgress < (m / baseM); mProgress++) {
            for (int nProgress = 0; nProgress < (n / baseN); nProgress++) {
                CopyIn(mProgress, nProgress);    // GM -> L1               搬运矩阵A，矩阵B，矩阵Bias
                SplitA();                        // L1 -> L0A              搬运矩阵A
                SplitB();                        // L1 -> L0B              搬运矩阵B
                SplitBias();                     // L1 -> BT               搬运矩阵Bias
                Compute();                       // L0A + L0B + BT -> L0C  进行矩阵计算
                CopyOut(mProgress, nProgress);   // L0C -> GM              搬运矩阵计算结果

                AscendC::CrossCoreSetFlag<2, PIPE_FIX>(SYNC_AIC_AIV_FLAG);   // 计算完一次分型，通知Vector可以计算
            }
        }
    }

private:
    __aicore__ inline uint32_t CeilCubeBlock(uint32_t len)
    {
        return (len + CUBE_BLOCK - 1) / CUBE_BLOCK;
    }

    __aicore__ inline void CopyIn(uint32_t mProgress, uint32_t nProgress)
    {
        AscendC::LocalTensor<half> a1Local = inQueueA1.AllocTensor<half>();
        AscendC::LocalTensor<half> b1Local = inQueueB1.AllocTensor<half>();
        AscendC::LocalTensor<half> bias1Local = inQueueC1.AllocTensor<half>();

        // 矩阵A: GM: [baseM, k], ND format => L1: [baseM, k], Nz format
        AscendC::Nd2NzParams nd2nzA1Params;
        nd2nzA1Params.ndNum = 1;
        nd2nzA1Params.nValue = baseM;
        nd2nzA1Params.dValue = k;
        nd2nzA1Params.srcNdMatrixStride = 0;
        nd2nzA1Params.srcDValue = k;
        nd2nzA1Params.dstNzC0Stride = CeilCubeBlock(baseM) * CUBE_BLOCK;
        nd2nzA1Params.dstNzNStride = 1;
        nd2nzA1Params.dstNzMatrixStride = 0;
        AscendC::DataCopy(a1Local, aGM[mProgress * baseM * k], nd2nzA1Params);

        // 矩阵B: GM: [k, baseN], ND format => L1: [k, baseN], Nz format
        AscendC::Nd2NzParams nd2nzB1Params;
        nd2nzB1Params.ndNum = 1;
        nd2nzB1Params.nValue = k;
        nd2nzB1Params.dValue = baseN;
        nd2nzB1Params.srcNdMatrixStride = 0;
        nd2nzB1Params.srcDValue = n;
        nd2nzB1Params.dstNzC0Stride = CeilCubeBlock(k) * CUBE_BLOCK;
        nd2nzB1Params.dstNzNStride = 1;
        nd2nzB1Params.dstNzMatrixStride = 0;
        AscendC::DataCopy(b1Local, bGM[nProgress * baseN], nd2nzB1Params);

        // 矩阵Bias: GM: [baseN], ND format => L1: [baseN], ND format
        AscendC::DataCopy(bias1Local, biasGM[nProgress * baseN], baseN);

        inQueueA1.EnQue(a1Local);
        inQueueB1.EnQue(b1Local);
        inQueueC1.EnQue(bias1Local);
    }

    __aicore__ inline void SplitA()
    {
        AscendC::LocalTensor<half> a1Local = inQueueA1.DeQue<half>();
        AscendC::LocalTensor<half> a2Local = inQueueA2.AllocTensor<half>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // 矩阵A: L1: [baseM, k], Nz format => L0A: [baseM, k], Nz format
        AscendC::LoadData2DParamsV2 loadDataParams;
        loadDataParams.mStartPosition = 0;
        loadDataParams.kStartPosition = 0;
        loadDataParams.mStep = CeilCubeBlock(baseM);
        loadDataParams.kStep = CeilCubeBlock(k);
        loadDataParams.srcStride = CeilCubeBlock(baseM);
        loadDataParams.dstStride = CeilCubeBlock(baseM);
        loadDataParams.ifTranspose = false;
        AscendC::LoadData(a2Local, a1Local, loadDataParams);
#else
        // 矩阵A: L1: [baseM, k], Nz format => L0A: [baseM, k], Zz format
        uint32_t dstOffset = CeilCubeBlock(k) * CUBE_BLOCK_SIZE;
        uint32_t srcOffset = CUBE_BLOCK_SIZE;

        AscendC::LoadData2DParams loadDataParams;
        loadDataParams.repeatTimes = CeilCubeBlock(k);
        loadDataParams.srcStride = CeilCubeBlock(baseM);
        loadDataParams.dstGap = 0;
        loadDataParams.ifTranspose = false;

        // 一行一行分形的遍历搬运
        for (int i = 0; i < CeilCubeBlock(baseM); ++i) {
            AscendC::LoadData(a2Local[i * dstOffset], a1Local[i * srcOffset], loadDataParams);
        }
#endif
        inQueueA2.EnQue<half>(a2Local);
        inQueueA1.FreeTensor(a1Local);
    }
    __aicore__ inline void SplitB()
    {
        AscendC::LocalTensor<half> b1Local = inQueueB1.DeQue<half>();
        AscendC::LocalTensor<half> b2Local = inQueueB2.AllocTensor<half>();

        // 矩阵B: L1: [k, baseN], Nz format => L0B: [k, baseN], Zn format
        uint32_t dstOffset = CUBE_BLOCK_SIZE;
        uint32_t srcOffset = CeilCubeBlock(k) * CUBE_BLOCK_SIZE;

        AscendC::LoadData2DParams loadDataParams;
        loadDataParams.repeatTimes = CeilCubeBlock(k);
        loadDataParams.srcStride = 1;
        loadDataParams.dstGap = CeilCubeBlock(baseN) - 1;
        loadDataParams.ifTranspose = true;

        // 一列一列分型的遍历搬运
        for (int i = 0; i < CeilCubeBlock(baseN); ++i) {
            AscendC::LoadData(b2Local[i * dstOffset], b1Local[i * srcOffset], loadDataParams);
        }

        inQueueB1.FreeTensor(b1Local);
        inQueueB2.EnQue<half>(b2Local);
    }

    __aicore__ inline void SplitBias()
    {
        AscendC::LocalTensor<half> bias1Local = inQueueC1.DeQue<half>();
        AscendC::LocalTensor<float> bias2Local = outQueueC2.AllocTensor<float>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        // __NPU_ARCH__为3510时，blockLen单位为32B
        AscendC::DataCopy(bias2Local, bias1Local, { 1, (uint16_t)(baseN * sizeof(float) / 32), 0, 0 });
#else
        // __NPU_ARCH__为2201时，blockLen单位为64B
        AscendC::DataCopy(bias2Local, bias1Local, { 1, (uint16_t)(baseN * sizeof(float) / 64), 0, 0 });
#endif

        outQueueC2.EnQue<float>(bias2Local);
        inQueueC1.FreeTensor(bias1Local);
    }
    __aicore__ inline void Compute()
    {
        AscendC::LocalTensor<half> a2Local = inQueueA2.DeQue<half>();
        AscendC::LocalTensor<half> b2Local = inQueueB2.DeQue<half>();
        AscendC::LocalTensor<float> bias2Local = outQueueC2.DeQue<float>();
        AscendC::LocalTensor<float> c1Local = outQueueCO1.AllocTensor<float>();

        AscendC::MmadParams mmadParams;
        mmadParams.m = baseM;
        mmadParams.n = baseN;
        mmadParams.k = k;
        mmadParams.cmatrixInitVal = false;
        AscendC::Mmad(c1Local, a2Local, b2Local, bias2Local, mmadParams);

        outQueueCO1.EnQue<float>(c1Local);
        inQueueA2.FreeTensor(a2Local);
        inQueueB2.FreeTensor(b2Local);
        outQueueC2.FreeTensor(bias2Local);
    }
    __aicore__ inline void CopyOut(uint32_t mProgress, uint32_t nProgress)
    {
        AscendC::LocalTensor<float> c1Local = outQueueCO1.DeQue<float>();

#if defined(__NPU_ARCH__) && (__NPU_ARCH__ == 3510)
        AscendC::FixpipeParamsArch3510<AscendC::CO2Layout::ROW_MAJOR> fixpipeParams;
#else
        AscendC::FixpipeParamsV220 fixpipeParams;
        fixpipeParams.ndNum = 1;
        fixpipeParams.srcNdStride = 0;
        fixpipeParams.dstNdStride = 0;
#endif
        fixpipeParams.nSize = baseN;
        fixpipeParams.mSize = baseM;
        fixpipeParams.srcStride = CeilCubeBlock(baseM) * CUBE_BLOCK;
        fixpipeParams.dstStride = n;     // dst GM matrix [m, n]
        AscendC::Fixpipe(cGM[nProgress * baseN + mProgress * baseM * n], c1Local, fixpipeParams);

        outQueueCO1.FreeTensor(c1Local);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::QuePosition::A1, 1> inQueueA1;
    AscendC::TQue<AscendC::QuePosition::A2, 1> inQueueA2;
    AscendC::TQue<AscendC::QuePosition::B1, 1> inQueueB1;
    AscendC::TQue<AscendC::QuePosition::B2, 1> inQueueB2;
    AscendC::TQue<AscendC::QuePosition::CO1, 1> outQueueCO1;
    AscendC::TQue<AscendC::QuePosition::C1, 1> inQueueC1;
    AscendC::TQue<AscendC::QuePosition::C2, 1> outQueueC2;

    AscendC::GlobalTensor<half> aGM;
    AscendC::GlobalTensor<half> bGM;
    AscendC::GlobalTensor<float> cGM;
    AscendC::GlobalTensor<half> biasGM;

    uint16_t m = 128, k = 128, n = 128;
    uint16_t baseM = 64, baseN = 64;
};

__global__ __mix__(1,1) void mmad_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR leakyreluRes,
    LeakyReLUCustomTilingData tiling)
{
    if ASCEND_IS_AIC {
        KernelMmad op;
        op.Init(a, b, bias, c);
        op.Process();
    } else { // ASCEND_IS_AIV
        KernelLeakyReLU op;
        op.Init(c, leakyreluRes, tiling);
        op.Process();
    }
}

#endif // MMAD_H

In [ ]:
%%writefile src/mmad_leakyrelu/leakyrelu.h
#ifndef LEAKYRELU_H
#define LEAKYRELU_H

#include "kernel_operator.h"

constexpr uint16_t SYNC_FLAG = 0;

struct LeakyReLUCustomTilingData
{
    uint32_t m;
    uint32_t n;
    uint32_t k;
    uint32_t baseM;
    uint32_t baseN;
    float alpha;
};

class KernelLeakyReLU {
public:
    __aicore__ inline KernelLeakyReLU() {}

    // numElem表示需要计算的元素个数
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR z, LeakyReLUCustomTilingData tiling)
    {
        this->m = tiling.m;
        this->n = tiling.n;
        this->k = tiling.k;
        this->baseM = tiling.baseM;
        this->baseN = tiling.baseN;
        this->leakyReluAlpha = tiling.alpha;
        uint32_t numElem = this->m * this->n;
        mmadResGm.SetGlobalBuffer((__gm__ float *)x, numElem);
        leakyreluResGm.SetGlobalBuffer((__gm__ float *)z, numElem);
        pipe.InitBuffer(inQueueX, 1, this->baseM * this->baseN * sizeof(float));
        pipe.InitBuffer(outQueueZ, 1, this->baseM * this->baseN * sizeof(float));
    }
    __aicore__ inline void Process()
    {
        uint32_t loopNum = (this->m / this->baseM) * (this->n / this->baseN);
        for (int32_t i = 0; i < loopNum; i++) {
            AscendC::CrossCoreWaitFlag(SYNC_FLAG);
            // 一行的矩阵子块中，属于第几块
            uint32_t cubePerRow = n / baseN;
            uint32_t rowIdx = i % cubePerRow;
            uint32_t rowStartIndex = baseN * rowIdx;
            // 多行的矩阵子块中，属于第几行
            uint32_t wholeRowidx = i / cubePerRow;
            uint32_t gmIndex = wholeRowidx * baseM * n + rowStartIndex;
            CopyIn(i, gmIndex);
            Compute(i);
            CopyOut(i, gmIndex);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress, uint32_t gmIndex)
    {
        AscendC::DataCopyParams copyParams;
        copyParams.blockCount = this->baseM;
        copyParams.blockLen = this->baseN * sizeof(float) / AscendC::ONE_BLK_SIZE;
        copyParams.srcGap = (this->n - this->baseN) * sizeof(float) / AscendC::ONE_BLK_SIZE;
        copyParams.dstGap = 0;

        AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();

        AscendC::DataCopy(xLocal, mmadResGm[gmIndex], copyParams);

        inQueueX.EnQue(xLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();

        AscendC::LeakyRelu(zLocal, xLocal, this->leakyReluAlpha, this->baseM * this->baseN);

        outQueueZ.EnQue<float>(zLocal);
        inQueueX.FreeTensor(xLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress, uint32_t gmIndex)
    {
        AscendC::DataCopyParams copyParams;
        copyParams.blockCount = this->baseM;
        copyParams.blockLen = this->baseN * sizeof(float) / AscendC::ONE_BLK_SIZE;
        copyParams.srcGap = 0;
        copyParams.dstGap = (this->n - this->baseN) * sizeof(float) / AscendC::ONE_BLK_SIZE;

        AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
        AscendC::DataCopy(leakyreluResGm[gmIndex], zLocal, copyParams);

        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> inQueueX;
    AscendC::TQue<AscendC::TPosition::VECOUT, 1> outQueueZ;
    AscendC::GlobalTensor<float> mmadResGm;
    AscendC::GlobalTensor<float> leakyreluResGm;
    uint32_t m, n, k;
    uint32_t baseM, baseN;
    float leakyReluAlpha;
};

#endif // LEAKYRELU_H

In [ ]:
%%writefile src/mmad_leakyrelu/mmad_leakyrelu.asc
#include <iostream>

#include "acl/acl.h"
#include "mmad.h"
#include "data_utils.h"

#define CHECK_ACL(x)                                                                        \
    do {                                                                                    \
        aclError __ret = x;                                                                 \
        if (__ret != ACL_ERROR_NONE) {                                                      \
            std::cerr << __FILE__ << ":" << __LINE__ << " aclError:" << __ret << std::endl; \
        }                                                                                   \
    } while (0);

int32_t main(int32_t argc, char *argv[])
{
    uint32_t M = 128;
    uint32_t N = 128;
    uint32_t K = 128;
    uint32_t baseM = 64;
    uint32_t baseN = 64;
    size_t aFileSize = M * K * sizeof(half);
    size_t bFileSize = K * N * sizeof(half);
    size_t biasFileSize = N * sizeof(half);
    size_t cFileSize = M * N * sizeof(float);
    uint32_t numBlocks = 1;                      // 启动的Aicore核数
    float alpha = 0.001;
    LeakyReLUCustomTilingData tiling = {M, N, K, baseM, baseN, alpha};

    CHECK_ACL(aclInit(nullptr));
    int32_t deviceId = 0;
    CHECK_ACL(aclrtSetDevice(deviceId));
    aclrtStream stream = nullptr;
    CHECK_ACL(aclrtCreateStream(&stream));

    uint8_t *aHost;
    uint8_t *aDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&aHost), aFileSize));
    CHECK_ACL(aclrtMalloc((void **)&aDevice, aFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/x1_gm.bin", aFileSize, aHost, aFileSize);
    CHECK_ACL(aclrtMemcpy(aDevice, aFileSize, aHost, aFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *bHost;
    uint8_t *bDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&bHost), bFileSize));
    CHECK_ACL(aclrtMalloc((void **)&bDevice, bFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/x2_gm.bin", bFileSize, bHost, bFileSize);
    CHECK_ACL(aclrtMemcpy(bDevice, bFileSize, bHost, bFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *biasHost;
    uint8_t *biasDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&biasHost), biasFileSize));
    CHECK_ACL(aclrtMalloc((void **)&biasDevice, biasFileSize, ACL_MEM_MALLOC_HUGE_FIRST));
    ReadFile("./input/bias_gm.bin", biasFileSize, biasHost, biasFileSize);
    CHECK_ACL(aclrtMemcpy(biasDevice, biasFileSize, biasHost, biasFileSize, ACL_MEMCPY_HOST_TO_DEVICE));

    uint8_t *cHost;
    uint8_t *cDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&cHost), cFileSize));
    CHECK_ACL(aclrtMalloc((void **)&cDevice, cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));

    uint8_t *leakyreluResHost;
    uint8_t *leakyreluResDevice;
    CHECK_ACL(aclrtMallocHost((void **)(&leakyreluResHost), cFileSize));
    CHECK_ACL(aclrtMalloc((void **)&leakyreluResDevice, cFileSize, ACL_MEM_MALLOC_HUGE_FIRST));

    mmad_leakyrelu_custom<<<numBlocks, nullptr, stream>>>(aDevice, bDevice, biasDevice, cDevice, leakyreluResDevice,
        tiling);
    CHECK_ACL(aclrtSynchronizeStream(stream));

    CHECK_ACL(aclrtMemcpy(leakyreluResHost, cFileSize, leakyreluResDevice, cFileSize, ACL_MEMCPY_DEVICE_TO_HOST));
    WriteFile("./output/output.bin", leakyreluResHost, cFileSize);

    CHECK_ACL(aclrtFree(aDevice));
    CHECK_ACL(aclrtFreeHost(aHost));
    CHECK_ACL(aclrtFree(bDevice));
    CHECK_ACL(aclrtFreeHost(bHost));
    CHECK_ACL(aclrtFree(biasDevice));
    CHECK_ACL(aclrtFreeHost(biasHost));
    CHECK_ACL(aclrtFree(cDevice));
    CHECK_ACL(aclrtFreeHost(cHost));
    CHECK_ACL(aclrtFree(leakyreluResDevice));
    CHECK_ACL(aclrtFreeHost(leakyreluResHost));

    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclFinalize());
    return 0;
}



用户可执行如下脚本，验证分块优化后的 CV 融合算子精度是否符合预期：

> **提示**：可尝试同时注释掉 `mmad.h` 中的 `CrossCoreSetFlag` 和 `leakyrelu.h` 中的 `CrossCoreWaitFlag`，体验缺少同步保障时的效果。

In [ ]:
# --npu-arch可以配置为dav-2201或dav-3510
!cd src/mmad_leakyrelu && bash run.sh --npu-arch=dav-2201

通过流水图对比可以看出：切分前 AIV 实际开始任务时间为 7.280，而切分后 AIV 开始时间提前至 5.256。切分后的版本让 AIV 无需长期等待 Cube 整体完成，Vector 的计算资源得以充分利用。流水线间尽可能并行是算子性能优化的关键方向。
<br>
<img src="./images/04_04_cv_fusion/4_pipe_flow_comparison.png" alt="turing_test"  width="1200px" >
<br>

---

# 5. 总结

<div style="text-align: left; float: left;">
<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">方案</td>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">AIC / AIV 交互</td>
    <td style="padding: 8px; border: 1px solid #ddd; font-weight: bold;">并行度</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">分步调用</td>
    <td style="padding: 8px; border: 1px solid #ddd;">无（串行）</td>
    <td style="padding: 8px; border: 1px solid #ddd;">无重叠</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">CV 融合（无切分）</td>
    <td style="padding: 8px; border: 1px solid #ddd;">整块计算完成后 AIC 通知 AIV 开始</td>
    <td style="padding: 8px; border: 1px solid #ddd;">AIV 启动前长时间闲置</td>
  </tr>
  <tr>
    <td style="padding: 8px; border: 1px solid #ddd;">CV 融合 + 切分</td>
    <td style="padding: 8px; border: 1px solid #ddd;">每次子块计算完成后AIC通知AIV</td>
    <td style="padding: 8px; border: 1px solid #ddd;">子块粒度流水并行</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

调试融合算子一般先从最简单的场景调起，分步调用虽然没有任何并行度，但是可以排除核间同步的影响，单独排查Cube和Vector侧的计算逻辑问题。等到精度正确后，可以逐步融合算子，添加核间同步逻辑，并且通过切分策略来提高流水并行度，获得更好的性能。


---
# 课后实践

前面的 `mmad_leakyrelu` 算子将矩阵切分为 2×2 = 4 个子块（baseM = 64, baseN = 64）。请自行实现一个 4×4 = 16 个子块版本（baseM = 32, baseN = 32），思考：切成更多的子块是否一定会提升性能？<br>

用户需修改如下3个文件：
1. 修改mmad.h 

In [ ]:
%%writefile src/04_04_cv_fusion/mmad.h

2. 修改leakyrelu.h

In [ ]:
%%writefile src/04_04_cv_fusion/leakyrelu.h

3. 修改mmad_leakyrelu.asc

In [ ]:
%%writefile src/04_04_cv_fusion/mmad_leakyrelu.asc

用户可以执行以下脚本验证精度是否符合预期：

In [ ]:
# --npu-arch可以配置为dav-2201或dav-3510
!cd src/04_04_cv_fusion && bash run.sh --npu-arch=dav-2201

用户可以执行如下命令来查看正确答案：

In [ ]:
!cat answer/04_04_cv_fusion/mmad.h

In [ ]:
!cat answer/04_04_cv_fusion/leakyrelu.h

In [ ]:
!cat answer/04_04_cv_fusion/mmad_leakyrelu.asc

通过多次分块实践可以发现：分块过小会增加调度开销（for 循环迭代过多导致标量流水过重），性能反而下降；分块过大则不利于流水并行，AIV 闲置时间变长。两者之间的均衡是算子优化中的一个重点，需要读者结合实际场景自行探索。